In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler, StandardScaler

CSV_PATH = r"outputs/results_ensemble_only_15-05-2026_21-50-38.csv"
df = pd.read_csv(CSV_PATH)
df["domain_match"] = df["domain_match"].astype(bool)

# --- normalisation helpers ---
def minmax(series):
    return pd.Series(
        MinMaxScaler().fit_transform(series.values.reshape(-1, 1)).flatten(),
        index=series.index,
    )

def standardise(series):
    return pd.Series(
        StandardScaler().fit_transform(series.values.reshape(-1, 1)).flatten(),
        index=series.index,
    )

# apply transforms
df["raw_conf_mm"]  = minmax(df["raw_confidence"])
df["raw_conf_std"] = standardise(df["raw_confidence"])
df["conf_mm"]      = minmax(df["confidence"])
df["conf_std"]     = standardise(df["confidence"])

COLORS = {True: "#2ecc71", False: "#e74c3c"}
LABELS = {True: "domain_match = True", False: "domain_match = False"}

print("Data loaded:", len(df), "rows")

## 1 & 2 — `raw_confidence`: Min-Max Normalised vs Standardised

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "raw_confidence — Min-Max Normalised",
        "raw_confidence — Standardised",
    ],
    horizontal_spacing=0.12,
)

for dm in [True, False]:
    subset = df[df["domain_match"] == dm]
    shared = dict(
        name=LABELS[dm],
        marker_color=COLORS[dm],
        boxmean="sd",
        legendgroup=str(dm),
    )
    fig.add_trace(
        go.Box(y=subset["raw_conf_mm"], **shared),
        row=1, col=1,
    )
    fig.add_trace(
        go.Box(y=subset["raw_conf_std"], showlegend=False, **{k: v for k, v in shared.items() if k != "showlegend"}, legendgroup=str(dm)),
        row=1, col=2,
    )

fig.update_layout(
    title_text="raw_confidence by domain_match",
    title_font_size=16,
    boxmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    height=500,
    template="plotly_white",
)
fig.update_yaxes(title_text="Normalised value [0, 1]", row=1, col=1)
fig.update_yaxes(title_text="Standardised value (z-score)", row=1, col=2)
fig.show()

## 3 & 4 — `confidence`: Min-Max Normalised vs Standardised

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "confidence — Min-Max Normalised",
        "confidence — Standardised",
    ],
    horizontal_spacing=0.12,
)

for dm in [True, False]:
    subset = df[df["domain_match"] == dm]
    shared = dict(
        name=LABELS[dm],
        marker_color=COLORS[dm],
        boxmean="sd",
        legendgroup=str(dm),
    )
    fig2.add_trace(
        go.Box(y=subset["conf_mm"], **shared),
        row=1, col=1,
    )
    fig2.add_trace(
        go.Box(y=subset["conf_std"], showlegend=False, **{k: v for k, v in shared.items() if k != "showlegend"}, legendgroup=str(dm)),
        row=1, col=2,
    )

fig2.update_layout(
    title_text="confidence by domain_match",
    title_font_size=16,
    boxmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
    height=500,
    template="plotly_white",
)
fig2.update_yaxes(title_text="Normalised value [0, 1]", row=1, col=1)
fig2.update_yaxes(title_text="Standardised value (z-score)", row=1, col=2)
fig2.show()

## Statistical Summary Tables — `raw_confidence`

In [ ]:
from IPython.display import display, HTML
from scipy import stats as sp_stats

def build_summary(col, label):
    rows = []
    for dm in [True, False]:
        s = df[df["domain_match"] == dm][col]
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        rows.append({
            "domain_match": str(dm),
            "n": len(s),
            "mean": round(s.mean(), 4),
            "std": round(s.std(), 4),
            "min": round(s.min(), 4),
            "Q1 (25%)": round(q1, 4),
            "median": round(s.median(), 4),
            "Q3 (75%)": round(q3, 4),
            "max": round(s.max(), 4),
            "IQR": round(q3 - q1, 4),
            "skewness": round(sp_stats.skew(s), 4),
            "kurtosis": round(sp_stats.kurtosis(s), 4),
        })
    summary = pd.DataFrame(rows).set_index("domain_match")
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    display(summary)
    return summary

# Table 1 — raw_confidence, Min-Max normalised
summary_mm = build_summary("raw_conf_mm", "raw_confidence  |  Min-Max Normalised  [0, 1]")

# Table 2 — raw_confidence, Standardised
summary_std = build_summary("raw_conf_std", "raw_confidence  |  Standardised (z-score)")